# Group B synthesis: what decision features add, against every alternative

Brings B1 (decision features), B2 (frozen embeddings, decision + E5, TAR, label-trained classifier) and B5 (native-text systems) together without redrawing the experiments' per-dataset panels: one ladder of every arm per task family (raw Δ), the same ladder on the pooled headroom share, a fold-level win-rate matrix over the main arms, and quality against compute. B3 (when it helps) and B4 (how much labelled data) keep their own summaries.

Common protocol: seven fixed predictors with library defaults and no tuning (linear/logistic, CatBoost, LightGBM, TabPFN-3.5, TabICL v2, LimiX-2, Kumo); the baseline is the same predictor on the non-text columns only; 18 datasets (3 binary, 6 multiclass, 9 regression) with the same five outer folds. Fixed headline metrics: binary ROC-AUC, multiclass macro one-vs-rest ROC-AUC, regression R². Datasets are the unit: each contributes its five-fold mean paired gain; raw gains stay within a task family (ΔAUC and ΔR², N = 9 each, separate panels) and only the headroom share (gain / (1 − baseline)) pools all 18 datasets. Means sit next to medians; intervals resample datasets (2,000 draws). Decision medians are taken over all 17 configured decision models; nothing selects a winner.

Reads the B1, B2 and B5 exports; each part waits while its export is absent.

In [1]:
import sys
from pathlib import Path
REPO_ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents) if (p / "pyproject.toml").is_file() and (p / "src").is_dir())
if str(REPO_ROOT) not in sys.path:
    sys.path.insert(0, str(REPO_ROOT))
import pandas as pd
from src.utils.notebook_display import display_json, display_table
from src.utils.notebook_report import begin_report, report_section, record, finish_report
from src.utils.provenance import provenance
from src.evaluation import benchmark_export, group_b, group_b_views as views, learner_scores, study_export
from src.visualize import critical_difference, figures, style
from src.visualize import group_b as gbfig
from src.visualize import group_b_synthesis as synthesis
style.apply()
NOTEBOOK = "B/B_synthesis"
begin_report()
save = figures.FigureSaver(NOTEBOOK)
report_section("1. What is complete: export, provenance and coverage")
display_json("Notebook execution identity", provenance())
EXPORTS = {study: benchmark_export.read(study) for study in ("B1", "B2", "B5")}
b1, b2, b5 = (EXPORTS[s][0] for s in ("B1", "B2", "B5"))
ROSTER = benchmark_export.decision_roster()
WAITING = "Waiting for export: no measurements are shown."
for study, (frames_, receipt_) in EXPORTS.items():
    print(f"{study}: {study_export.describe(receipt_)}")
    record({k: receipt_.get(k) for k in ("status", "path", "created_utc", "origin", "sha256", "bytes", "schema", "receipt_sha256")}, f"{study} export identity")
    if frames_:
        display_table(f"{study}: planned versus complete five-fold arms per phase and predictor", frames_["coverage"], preview_rows=14)
        if "median_roster_gaps" in frames_ and not frames_["median_roster_gaps"].empty:
            display_table(f"{study}: dataset x predictor cells whose median lacks configured decision models", frames_["median_roster_gaps"])

B1: Waiting for export: C:\Users\U0152019\PhD Documents\Projects\5. JEVPFN\JEVPFN\output_JEVPFN\B\exp04_decision_features\results\group_b\b1_decision_features.zip is absent. No measurements are shown.
B2: Waiting for export: C:\Users\U0152019\PhD Documents\Projects\5. JEVPFN\JEVPFN\output_JEVPFN\B\exp05_frozen_embeddings\results\group_b\b2_representations.zip is absent. No measurements are shown.
B5: Waiting for export: C:\Users\U0152019\PhD Documents\Projects\5. JEVPFN\JEVPFN\output_JEVPFN\B\exp07_native_text\results\group_b\b5_native_text.zip is absent. No measurements are shown.


## 2. Summary: the representation ladder

Rows run from decision features (per column, joint, both; median model) through decision + E5-small, frozen embeddings and label-trained routes to native systems; one marker shape per predictor, filled = mean, open = median, lines = dataset-bootstrap 95% intervals. Native systems are compared with the median no-text baseline over the seven predictors (they share no predictor).

In [2]:
report_section("2. Summary: the representation ladder")
ladder = views.ladder(b1, b2, b5)
if ladder.empty:
    print(WAITING)
else:
    gbfig.render(save, synthesis.representation_ladder(ladder, name="b_ladder_raw_gain", what="Every Group B arm"))
    display_table("Ladder: per arm and predictor, mean/median gain per task family and pooled headroom share", ladder, preview_rows=24)

Waiting for export: no measurements are shown.


## 3. The same ladder on the headroom share (all 18 datasets)

The only scale that pools classification and regression. Holm/Bayesian margins are added only once the framework is approved (see `general/5_synthesis`).

In [3]:
report_section("3. The ladder on the headroom share")
if ladder.empty:
    print(WAITING)
else:
    gbfig.render(save, synthesis.representation_ladder(ladder, name="b_ladder_headroom_share", what="Every Group B arm", share=True))
    print(f"Statistical framework: {learner_scores.framework_state()['framework_status']}; no confirmatory margin is drawn.")

Waiting for export: no measurements are shown.


## 4. Fold-level win rates between the main arms

Share of (dataset, predictor, fold) cells on which one arm beats another; descriptive, dataset-cluster bootstrap intervals in the table.

In [4]:
report_section("4. Fold-level win rates between the main arms")
scores = views.main_arm_fold_scores(b1, b2, roster=ROSTER)
if scores.empty:
    print(WAITING)
else:
    pairwise = views.pairwise_win_rates(scores, list(views.MAIN_ARMS))
    gbfig.render(save, synthesis.win_rate_matrix(pairwise, name="b_win_rate_matrix", what="Main Group B arms"))
    display_table("Pairwise fold-level win rates (predictors pooled)", pairwise, preview_rows=30)

Waiting for export: no measurements are shown.


## 5. Quality against compute

Mean headroom share against measured extraction GPU-hours (Slurm compute ledger and the 0A token audit) plus predictor fit time per 1,000 rows. Arms without a measured cost are listed, never estimated.

In [5]:
import json
from src.evaluation import model_properties
from src.utils import paths
report_section("5. Quality against compute")
if not b1:
    print(WAITING)
else:
    ledger_path = paths.outputs_dir() / "compute_ledger.json"
    audit_path = paths.manifests_dir("experiment_0") / "before_extraction" / "token_audit.json"
    ledger = json.loads(ledger_path.read_text(encoding="utf-8")) if ledger_path.is_file() else None
    audit = json.loads(audit_path.read_text(encoding="utf-8")) if audit_path.is_file() else None
    extraction = model_properties.compute_per_thousand(ledger, audit, ROSTER)
    display_table("Decision-feature extraction GPU-hours per 1,000 distinct requests", extraction, preview_rows=17)
    costs = views.compute_costs(b1["folds"], rows=views.dataset_rows(), extraction=extraction, roster=ROSTER)
    single = b1["summary"][b1["summary"].representation.str.startswith("decision/both/")]
    points = views.pareto_points([ladder, single], costs)
    gbfig.render(save, synthesis.quality_vs_compute(points, name="b_quality_vs_compute", what="Quality against compute"))
    display_table("Quality and compute per arm (missing cost = not measured)", points, preview_rows=24)

Waiting for export: no measurements are shown.


In [6]:
finish_report(NOTEBOOK, save.summary() + "\nOnly the verified exports and local caches named above were read; no inference, model training or VSC changes were performed.")

B/B_synthesis

1. What is complete: export, provenance and coverage


Notebook execution identity
{
  "code_sha256": "41477e70c06f4bea09495570fa55522fe62d639d69a854cd48125d8fa3d2c814",
  "git_head": "ac34e04f34066a8bc54cc08092ca38ea3125314b",
  "git_dirty": true,
  "tfm_library_pin": "4574eea0c2ce6646b5d2e9c7f53a6ef3882c368c dm-library (remotes/origin/HEAD)\n 248f137046487e0d57360abba1e060420ae54e1a tfm-library (remotes/origin/HEAD)\n 2208cf68d6351eff2cb0ddb5a9eb1d822523457b vsc-library (heads/main-8-g2208cf6)",
  "environment": {
    "python": "3.12.10",
    "numpy": "2.5.4",
    "pandas": "2.3.3",
    "PyYAML": "6.0.3",
    "matplotlib": "3.11.2",
    "pyarrow": "25.0.1"
  }
}


B1 export identity
{
  "status": "awaiting_export",
  "path": "C:\\Users\\U0152019\\PhD Documents\\Projects\\5. JEVPFN\\JEVPFN\\output_JEVPFN\\B\\exp04_decision_features\\results\\group_b\\b1_decision_features.zip",
  "created_utc": null,
  "origin": null,
  "sha256": null,
  "bytes": null,
  "schema": null,
